In [1]:
from langchain.document_loaders.csv_loader import CSVLoader

In [2]:
loader=CSVLoader(file_path="bcit_faqs.csv",source_column="prompt")
docs=loader.load()
docs[:5]

[Document(metadata={'source': 'I have never done programming in my life. Can I take this bootcamp?', 'row': 0}, page_content='prompt: I have never done programming in my life. Can I take this bootcamp?\nresponse: Yes, this is the perfect bootcamp for anyone who has never done coding and wants to build a career in the IT/Data Analytics industry or just wants to perform better in your current job or business using data.'),
 Document(metadata={'source': 'Why should I trust bcitworld?', 'row': 1}, page_content='prompt: Why should I trust bcitworld?\nresponse: Till now 9000 + learners have benefitted from the quality of our courses. You can check the review section and also we have attached their LinkedIn profiles so that you can connect with them and ask directly.'),
 Document(metadata={'source': 'Is there any prerequisite for taking this bootcamp ?', 'row': 2}, page_content='prompt: Is there any prerequisite for taking this bootcamp ?\nresponse: Our bootcamp is specifically designed for b

In [3]:
import google.generativeai as genai
import streamlit as st
import os
from google_api_secret import api_key

genai.configure(api_key=api_key)

In [4]:
# pip install InstructorEmbedding
# pip install torch
# pip install -U sentence-transformers==2.2.2
# pip install faiss-cpu==1.7.4 for vector database
import torch
from langchain.embeddings import HuggingFaceInstructEmbeddings
from langchain.vectorstores import FAISS

instructor_embeddings=HuggingFaceInstructEmbeddings()
vectordb=FAISS.from_documents(documents=docs,embedding=instructor_embeddings)

C:\Python311\Lib\site-packages\pydantic\_internal\_fields.py:161: UserWarning: Field "model_name" has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
C:\Python311\Lib\site-packages\IPython\core\interactiveshell.py:3526: LangChainDeprecationWarning: Default values for HuggingFaceInstructEmbeddings.model_name were deprecated in LangChain 0.2.16 and will be removed in 0.4.0. Explicitly pass a model_name to the HuggingFaceInstructEmbeddings constructor instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


load INSTRUCTOR_Transformer


C:\Python311\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
C:\Python311\Lib\site-packages\sentence_transformers\models\Dense.py:63: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to 

max_seq_length  512


In [5]:
retriever=vectordb.as_retriever()
rdocs=retriever.get_relevant_documents("how about job placement support?")
rdocs

C:\Users\my lapi\AppData\Local\Temp\ipykernel_9476\3418398517.py:2: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  rdocs=retriever.get_relevant_documents("how about job placement support?")


[Document(metadata={'source': 'Do you provide any job assistance?', 'row': 11}, page_content='prompt: Do you provide any job assistance?\nresponse: Yes, We help you with resume and interview preparation along with that we help you in building online credibility, and based on requirements we refer candidates to potential recruiters.'),
 Document(metadata={'source': 'Will this course guarantee me a job?', 'row': 33}, page_content='prompt: Will this course guarantee me a job?\nresponse: We created a much lighter version of this course on YouTube available for free (click this link) and many people gave us feedback that they were able to fetch jobs (see testimonials). Now this paid course is at least 5x better than the YouTube course which gives us ample confidence that you will be able to get a job. However, we want to be honest and do not want to make any impractical promises! Our guarantee is to prepare you for the job market by teaching the most relevant skills, knowledge & timeless pr

In [6]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains.question_answering import load_qa_chain
from langchain.prompts import PromptTemplate

In [7]:
prompt_template="""
Answer the question from the provided context, make sure to provide all the details, if the answer is not in the provided context then just say,
"I don't know the answer, please ask another question", don't provide the wrong answer.

CONTEXT:{context}
QUESTION:{question}

"""

In [8]:
llm=ChatGoogleGenerativeAI(model="gemini-pro",temperature=0.3,api_key=api_key)
prompt=PromptTemplate(template=prompt_template,input_variables=["context","question"])
chain=load_qa_chain(llm,chain_type="stuff",prompt=prompt)

C:\Users\my lapi\AppData\Local\Temp\ipykernel_9476\445874687.py:3: LangChainDeprecationWarning: This class is deprecated. See the following migration guides for replacements based on `chain_type`:
stuff: https://python.langchain.com/v0.2/docs/versions/migrating_chains/stuff_docs_chain
map_reduce: https://python.langchain.com/v0.2/docs/versions/migrating_chains/map_reduce_chain
refine: https://python.langchain.com/v0.2/docs/versions/migrating_chains/refine_chain
map_rerank: https://python.langchain.com/v0.2/docs/versions/migrating_chains/map_rerank_docs_chain

See also guides on retrieval and question-answering here: https://python.langchain.com/v0.2/docs/how_to/#qa-with-rag
  chain=load_qa_chain(llm,chain_type="stuff",prompt=prompt)


In [11]:
def user_input(user_question):
    retriever=vectordb.as_retriever()
    docs=retriever.get_relevant_documents(user_question)
    
    response=chain(
       {"input_documents":docs,"question":user_question}
       ,return_only_outputs=True)
    
    return response

In [12]:
user_input("do you have emi options?")

{'output_text': 'No'}